# Assignment 4 — Sequential knowledge editing experiment (Kaggle runner)

**Before running:** right panel → *Session options* → Accelerator **GPU T4 x2**, Internet **On** (Internet requires a phone-verified Kaggle account). The GitHub repository must be **public**.

**Order of work** (setup — prefilter + covariance statistics — runs automatically whenever it is missing)
1. `MODE = "pilot"` — interactive: press **Run All**. ≈30–60 min. Runs 5 edits per method on the **dev** split and prints seconds per edit. Send these numbers to the team, fix `n_max` in the protocol freeze form, **upload the form to Moodle**, then switch the session off (power button) to save GPU quota.
2. `MODE = "main"` — final runs, in the background: **Save Version → Save & Run All (Commit)**; the browser can be closed.
   Each session stops itself after `HOURS` hours. To continue: *Add Input → Your Work →* this notebook's latest version, then commit again; runs resume from their last checkpoint.
3. `MODE = "analysis"` — CPU is enough. Add the outputs of the final gpt2-xl and llama3.2-3b versions as inputs and press Run All: tables, tests, figures in `results/analysis/`.


In [ ]:
# ======== EDIT THESE ========
REPO_URL = "https://github.com/YOUR_GITHUB/YOUR_REPO.git"
BRANCH   = "main"
MODEL    = "gpt2-xl"          # "gpt2-xl" or "llama3.2-3b"  (one model per Kaggle account)
MODE     = "pilot"            # "pilot" | "main" | "analysis"
HOURS    = 10.5               # each GPU queue stops after this many hours (Kaggle session limit is 12 h)
# Final-run job lists (model:method:seed[:single]); one queue per T4 GPU, balanced to ~20 GPU-hours each (gpt2-xl pilot timing)
JOBS_GPU0 = [f"{MODEL}:ROME:0", f"{MODEL}:ROME:1", f"{MODEL}:MEMIT:2", f"{MODEL}:FT:0", f"{MODEL}:none:0"]
JOBS_GPU1 = [f"{MODEL}:MEMIT:0", f"{MODEL}:MEMIT:1", f"{MODEL}:ROME:2", f"{MODEL}:FT:1", f"{MODEL}:FT:2",
             f"{MODEL}:none:1", f"{MODEL}:none:2",
             f"{MODEL}:ROME:0:single", f"{MODEL}:MEMIT:0:single", f"{MODEL}:FT:0:single"]


In [ ]:
import os, subprocess, glob, json, shutil
W = "/kaggle/working/ke-seq"
if not os.path.exists(W):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, W], check=True)
else:
    subprocess.run(["git", "-C", W, "pull"], check=False)
os.chdir(W)
print("commit:", subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q "transformers==5.5.4" "datasets==4.8.5" "huggingface_hub==1.17.0"
!python -c "import torch, transformers, datasets; print('torch', torch.__version__, 'cuda', torch.version.cuda, '| transformers', transformers.__version__, '| datasets', datasets.__version__, '| GPUs', torch.cuda.device_count())"


In [ ]:
def restore(sub):
    for src in sorted(glob.glob(f"/kaggle/input/**/ke-seq/{sub}", recursive=True)):
        print("restoring", src)
        shutil.copytree(src, sub, dirs_exist_ok=True)
for sub in ["results", "data/stats"]:
    restore(sub)
for src in glob.glob("/kaggle/input/**/ke-seq/data/processed/ppl_passages.jsonl", recursive=True)[:1]:
    if not os.path.exists("data/processed/ppl_passages.jsonl"):
        shutil.copy(src, "data/processed/")
!cd data/processed && sha256sum -c SHA256SUMS

In [ ]:
def setup(model):
    if not os.path.exists(f"results/{model}/prefilter.json"):
        !python src/prefilter.py --model {model}
    else:
        print("prefilter.json exists:", json.load(open(f"results/{model}/prefilter.json"))["summary"])
    !python src/compute_cov.py --model {model}

if MODE in ("pilot", "main"):
    setup(MODEL)

In [ ]:
if MODE == "pilot":
    for m in ["none", "FT", "ROME", "MEMIT"]:
        print(f"===== pilot {MODEL} {m} =====", flush=True)
        !python src/run_experiment.py --model {MODEL} --method {m} --seed 0 --split dev --n_max 5 --overwrite 2>&1 | tail -4
    print("\n===== PILOT SUMMARY (send this to the team) =====")
    for d in sorted(glob.glob(f"results/{MODEL}/seq/*_dev")):
        print(os.path.basename(d), json.load(open(d + "/run_summary.json")))

In [ ]:
if MODE == "main":
    p0 = subprocess.Popen(["python", "src/run_queue.py", "--gpu", "0", "--hours", str(HOURS), "--jobs", *JOBS_GPU0])
    p1 = subprocess.Popen(["python", "src/run_queue.py", "--gpu", "1", "--hours", str(HOURS), "--jobs", *JOBS_GPU1])
    p0.wait(); p1.wait()
    !python src/error_export.py --model {MODEL}

In [ ]:
# Progress of every run
for d in sorted(glob.glob("results/*/*/*_seed*")):
    prog = json.load(open(d + "/progress.json")) if os.path.exists(d + "/progress.json") else {}
    summ = json.load(open(d + "/run_summary.json")) if os.path.exists(d + "/run_summary.json") else {}
    print(f"{d:50s} done={prog.get('done')}  {summ}")

In [ ]:
if MODE == "analysis":
    !python src/analyze.py
    !cat results/analysis/main_results.md